# Banking Intent — Fine-tune Qwen3-8B on Kaggle

**Before running:**
1. Kaggle → Add-ons → **Secrets**: add `HF_TOKEN` (HuggingFace token with write access)
2. Notebook settings → **Accelerator**: GPU T4 x2 or P100
3. Notebook settings → **Internet**: On
4. Fill in `GITHUB_REPO_URL` and `HF_REPO_ID` in the next cell

In [ ]:
# === CONFIG — change these two lines ===
GITHUB_REPO_URL = "https://github.com/nguyenvmthien/banking-intent-unsloth.git"
HF_REPO_ID      = "minhthien/banking-intent-unsloth"
# ========================================

import os
WORKING_DIR = f"/kaggle/working/{GITHUB_REPO_URL.rstrip('/').split('/')[-1].removesuffix('.git')}"
print(f"Repo will be cloned to: {WORKING_DIR}")

## 1. Environment setup

In [ ]:
import subprocess, sys

def run(cmd, capture=True):
    result = subprocess.run(cmd, shell=True, capture_output=capture, text=True)
    if result.returncode != 0:
        print(result.stderr[-3000:] if capture else "")
        raise RuntimeError(f"Command failed: {cmd}")
    return result.stdout if capture else ""

# Unsloth must be installed before transformers to avoid version conflicts
run('pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"')
run('pip install -q trl peft datasets langsmith huggingface_hub pandas scikit-learn tqdm pyyaml')
print("Dependencies installed.")

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
print("HF_TOKEN loaded.")

## 2. Clone repo and prepare data

In [ ]:
run(f'git clone {GITHUB_REPO_URL} {WORKING_DIR}')
os.chdir(f"{WORKING_DIR}/scripts")
print(f"Repo cloned to {WORKING_DIR}")

In [ ]:
run('python preprocess_data.py')
print("Data ready.")

## 3. Configure HF repo for checkpoint push

In [ ]:
import yaml

config_path = f"{WORKING_DIR}/configs/train.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

config['hub_model_id'] = HF_REPO_ID

with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print(f"hub_model_id set to: {HF_REPO_ID}")

## 4. Train

If this cell is re-run after a session restart, it will **automatically resume** from the latest local checkpoint (if any), or pull from Hub if disk was wiped.

In [ ]:
# If Kaggle disk was wiped, restore latest checkpoint from Hub before training
import glob
from huggingface_hub import snapshot_download

output_dir = f"{WORKING_DIR}/outputs/checkpoint"
os.makedirs(output_dir, exist_ok=True)

has_local_checkpoint = bool(glob.glob(os.path.join(output_dir, "checkpoint-*")))

if not has_local_checkpoint:
    try:
        print(f"No local checkpoint. Trying to restore from Hub: {HF_REPO_ID}")
        snapshot_download(
            repo_id=HF_REPO_ID,
            local_dir=output_dir,
            token=os.environ["HF_TOKEN"],
        )
        print("Restored from Hub.")
    except Exception as e:
        print(f"Hub restore skipped ({e}) — starting fresh.")
else:
    latest = sorted(glob.glob(os.path.join(output_dir, "checkpoint-*")))[-1]
    print(f"Local checkpoint found: {latest}")

In [ ]:
os.chdir(f"{WORKING_DIR}/scripts")

# Run training with live output (no capture)
train_result = subprocess.run([sys.executable, "train.py"], env=os.environ)
if train_result.returncode != 0:
    raise RuntimeError("Training failed — check logs above.")
print("Training complete.")

## 5. Quick sanity check — predict one sample

In [ ]:
import sys
sys.path.insert(0, f"{WORKING_DIR}/scripts")

from inference import IntentClassification

clf = IntentClassification(f"{WORKING_DIR}/configs/inference.yaml", mode="finetuned")
test_input = "I lost my credit card, how do I order a replacement?"
result = clf.predict(test_input)
print(result)

## 6. Evaluate on test set

Chạy cả 3 mode và in bảng so sánh accuracy — đúng theo yêu cầu đề (zero-shot / few-shot / fine-tuned).

In [ ]:
os.chdir(f"{WORKING_DIR}/scripts")

eval_result = subprocess.run(
    [sys.executable, "evaluate.py", "--mode", "all", "--few_shot_k", "5"],
    env=os.environ,
)
if eval_result.returncode != 0:
    raise RuntimeError("Evaluation failed — check logs above.")